## RNN

In [1]:
# RNN 이 EmbeddingMLP와 다른 점 => 단어를 순서대로 읽음
# EmbeddingMLP  : ['영화', '좋다', '별로'] -> 평균 => 벡터 1개 (순서 무시)
# RNN           : '영화' -> '좋다' -> '별로'
# => 한 단어씩 읽으면서 이전 결과를 다음에 전달(hidden_state)

In [ ]:
# 1. 데이터 로드 & 전처리
# 2. 토크나이저 (한국어)
# 3. 단어 사전 구축
# 4. 문장 → 시퀀스 변환
# 5. 패딩
# 6. DataLoader
# 7. RNN 모델
# 8. 학습
# 9. 평가
# 10. 예측 함수

In [3]:
import pandas as pd
import re

df = pd.read_csv('daum_movie_review.csv')
df['target'] = df['rating'].apply(lambda x : 1 if x > 5 else 0)
df['clean'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]', '', str(x)))

# 5 기준, 긍정 = 1

In [4]:
# 토크나이저
from konlpy.tag import Okt
okt = Okt()

def kor_tokenizer(text):
    return [
        word for word, pos in okt.pos(text, stem=True) 
        if pos in ['Noun', 'Verb', 'Adjective']
        and len(word) >= 2
    ]

In [5]:
# 단어 사전 구축
from collections import Counter

vocab = Counter()
for text in df['clean']:
    vocab.update(kor_tokenizer(text))
    # Counter.update()
    
# 일반 dict.update() → 덮어씌움
# d = {'영화': 2}
# d.update({'영화': 3})
# → {'영화': 3}  ← 3으로 덮어씌워짐
# Counter.update() → 누적
# c = Counter({'영화': 2})
# c.update({'영화': 3})
# → Counter({'영화': 5})  ← 2 + 3 = 5로 누적

vocab_size = 10000

word_to_index = {
    word : idx + 2 for idx, (word, count) 
    in enumerate(vocab.most_common(vocab_size))
}

word_to_index['<PAD>'] = 0
word_to_index['<UNK>'] = 1

In [6]:
# 문장 -> 시퀀스 변환
def word2Sequence(text):
    return [
        word_to_index.get(word, 1) for word in kor_tokenizer(text)
    ]

In [7]:
# 패딩(Padding)
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

MAX_LEN = 500

def padding(texts):
    result = []
    for text in texts:
        seq = word2Sequence(text)   # 문장 -> 인덱스 리스트
        seq = seq[:MAX_LEN]         # 500개 넘으면 자르기
        seq = torch.LongTensor(seq) # 텐서로 변환
        result.append(seq)

    return pad_sequence(
        result,
        batch_first=True,
        padding_value=0
    )

# 지난번 : 무조건 MAX_LEN(500) 길이로 맞춤
# seq = encoded + [0] * (max_len - len(encoded))
# → 모든 문장이 길이 500

# 이번 : 배치 안에서 가장 긴 문장 기준으로 맞춤
# pad_sequence(result, batch_first=True, padding_value=0)
# → 배치 안에서 가장 긴 문장 길이로만 맞춤

In [8]:
# DataLoader
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

x = padding(df['clean'])
y = torch.FloatTensor(df['target'].values)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state=42)

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)
# TensorDataset 텐서 두 개를 묶어주는 간단한 Dataset
# 텐서로 변환된 데이터가 있으면 바로 쓸 수 있음


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [9]:
# RNN 모델
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # 단어를 순서대로 읽음 (순서 유지)
        self.rnn = nn.RNN(
            input_size = embedding_dim, # 입력 벡터 차원 (embedding 크기)   
            hidden_size = hidden_dim,   # hidden state 크기
            batch_first = True          # (B, L, D) 순서로 입력받음
        )

        self.fc = nn.Linear(hidden_dim, 1)

# RNN은 단어를 하나씩 읽으면서 hidden state를 계속 업데이트
# 이전 hidden state가 다음 단계로 전달되면서 문맥이 누적

    def forward(self, x):
        x = self.embedding(x)           # B L -> B L D
        output, hidden = self.rnn(x)    
        # output : B L H => 모든 시점의 hidden state
        # hidden : 1 B H => 마지막 시점의 hidden state만
        # 문장 전체를 읽고 나서 판단하니까 마지막 hidden만 씀, output 무시
        hidden = hidden.squeeze(0)      # (1, B, H) -> (B, H)
        logits = self.fc(hidden)        # (B, H) -> (B, 1)
        return logits

# B : Batch size    : 한 번에 처리하는 문장 수 (64)
# L : Length        : 문장 길이, 단어 수 (500)
# D : Dimension     : 임베딩 차원 (128)
# H : Hidden size   : hidden state 크기 (64)

In [10]:
# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# 모델 생성
model = RNNClassifier(
    vocab_size=vocab_size + 2,
    embedding_dim=128,
    hidden_dim=64
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 학습
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch).squeeze(1)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1} Loss : {total_loss/len(train_loader):.4f}')

Epoch 1 Loss : 0.5578
Epoch 2 Loss : 0.5538
Epoch 3 Loss : 0.5507
Epoch 4 Loss : 0.5504
Epoch 5 Loss : 0.5523


In [11]:
# 평가
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        logits = model(X_batch).squeeze(1)
        pred = torch.sigmoid(logits) >= 0.5
        correct += (pred == y_batch).sum().item()
        total += y_batch.size(0)
print(f'Accuracy : {correct/total:.4f}')

# 예측 함수
def predict_sentiment(text):
    model.eval()
    seq = word2Sequence(text)
    seq = seq[:MAX_LEN]
    seq = torch.LongTensor(seq).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(seq)
        prob = torch.sigmoid(logits)
    return prob.item()

Accuracy : 0.7576


In [12]:
# 테스트
text = "진짜 너무 재미있고 감동적인 영화였다"
score = predict_sentiment(text)
print(score)
print("긍정" if score >= 0.5 else "부정")

0.6338497400283813
긍정


## LSTM

In [13]:
# 데이터 준비
# step 1. 데이터 로드 & 전처리
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from nltk.corpus import movie_reviews

# 데이터 로드
reviews = [ movie_reviews.raw(fileid) for fileid in movie_reviews.fileids() ]
categories = [ movie_reviews.categories(fileid)[0] for fileid in movie_reviews.fileids() ]

np.random.seed(42)
tf.random.set_seed(42)


In [14]:
# step 2. Tokenizer
max_words = 10000

# Keras Tokenizer : 빈도 높은 10000개 단어만 선택
tokenizer = Tokenizer(num_words=max_words, oov_token='UNK')

# 단어 인덱스 구축
tokenizer.fit_on_texts(reviews)

# 문장 -> 인덱스 시퀀스
x = tokenizer.texts_to_sequences(reviews)

In [15]:
# step 3. Padding
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = 500

# truncating='pre' : 앞부분을 자름
x = pad_sequences(x, maxlen=maxlen, truncating='pre')

# 앞부분을 자르는 이유 => 리뷰는 보통 결론이 뒤에 있어서

In [16]:
# step 4. change label
label_dict = {'pos': 1, 'neg' : 0}
y = np.array([label_dict[c] for c in categories])

In [17]:
# step 5. train/test split
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, random_state=42,
    test_size=0.2
)

print(x_train.shape)

(1600, 500)


In [18]:
# step 6. DataLoader
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

x_train = torch.LongTensor(x_train)
x_test = torch.FloatTensor(x_test)
y_train = torch.LongTensor(y_train)
y_test = torch.FloatTensor(y_test)

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [24]:
# LSTM Model
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding= nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size = embedding_dim,
            hidden_size = hidden_dim,
            batch_first = True
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)                   # (B, L) -> (B, L, D)
        ouput, (hidden, cell) = self.lstm(x)    # output: (B, L, H), hidden: (1, B, H)
        hidden = hidden.squeeze(0)              # (1, b, h) -> (B, H)
        return self.fc(hidden)                  # (b, H) -> (B, 1)
    
# 감성 분류에서는 cell 쓰지 않음 => 분류 문제에는 요약본(hidden)으로 충분

In [28]:
# 모델 생성 & 학습
# 모델 생성
model_lstm = LSTMClassifier(
    vocab_size = max_words,
    embedding_dim = 64,
    hidden_dim = 64
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=1e-4)

# 학습
def train_model(model, loader, optimizer, criterion, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.float().to(device)
            optimizer.zero_grad()
            logits = model(x_batch).squeeze(1)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}: Loss {total_loss/len(loader):.4f}")

train_model(model_lstm, train_loader, optimizer, criterion, epochs=8)

Epoch 1: Loss 0.6955
Epoch 2: Loss 0.6933
Epoch 3: Loss 0.6916
Epoch 4: Loss 0.6898
Epoch 5: Loss 0.6881
Epoch 6: Loss 0.6863
Epoch 7: Loss 0.6845
Epoch 8: Loss 0.6827


In [33]:
# 모델 평가
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.long().to(device), y_batch.float().to(device)

            outputs = model(x_batch).squeeze(1)
            outputs = torch.sigmoid(outputs)

            correct += ((outputs > 0.5).float() == y_batch).sum().item()
            total += y_batch.size(0)

    return correct / total

print(f"Test Accuracy (LSTM): {evaluate(model_lstm, test_loader):.4f}")

Test Accuracy (LSTM): 0.5050


In [40]:
# BiLSTM
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True # 이것만 추가됨
        )

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),  # hidden_dim * 2 => 양방향
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        x = self.embedding(x)
        _, (hn, _) = self.lstm(x)
        # hn : (2, B, H) -> 정방향, 역방향 각각의 마지막 hidden : 2
        x = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1) # (B * H*2)
        # hn[-2] : 정방향 마지막 hidden
        # hn[-1] : 역방향 마지막 hidden
        # torch.cat → (B, H*2) 로 합침
        return self.fc(x)

In [41]:
model_bilstm = BiLSTMModel(
    vocab_size=max_words,
    embed_dim=64,
    hidden_dim=64
).to(device)

optimizer_bilstm = torch.optim.Adam(model_bilstm.parameters(), lr=1e-4)

train_model(model_bilstm, train_loader, optimizer_bilstm, criterion, epochs=8)

print(f'Test Accuracy (BiLSTM): {evaluate(model_bilstm, test_loader):.4f}')

Epoch 1: Loss 0.7181
Epoch 2: Loss 0.7139
Epoch 3: Loss 0.7083
Epoch 4: Loss 0.7024
Epoch 5: Loss 0.6978
Epoch 6: Loss 0.6939
Epoch 7: Loss 0.6909
Epoch 8: Loss 0.6891
Test Accuracy (BiLSTM): 0.5025
